In [ ]:
# Dual streaming loader

import pymysql
from pathlib import Path
import os

# Masking my Password
import getpass

# Prompt the user for a password
password = getpass.getpass("Enter your password: ")

BASE = Path(r"C:\ProgramData\MySQL\MySQL Server 8.0\Uploads\energy_project_data")

LOADERS = {
    "demand": {
        "dir": BASE / "demand_chunks",
        "sql": """
LOAD DATA LOCAL INFILE '{path}'
INTO TABLE stg_demand
CHARACTER SET utf8mb4
FIELDS TERMINATED BY ','
ENCLOSED BY '"'
LINES TERMINATED BY '\\r\\n'
IGNORE 1 LINES
(MeasureItem, DateUTC, DateShort, TimeFrom, TimeTo, CountryCode,
 Cov_ratio, Value, Value_ScaleTo100, year);
"""
    },
    "price": {
        "dir": BASE / "price_chunks",
        "sql": """
LOAD DATA LOCAL INFILE '{path}'
INTO TABLE stg_price
CHARACTER SET utf8mb4
FIELDS TERMINATED BY ','
ENCLOSED BY '"'
LINES TERMINATED BY '\\r\\n'
IGNORE 1 LINES
(Country, ISO3_Code, Datetime_UTC, Datetime_Local, Price_EUR_MWhe);
"""
    }
}

def main():
    conn = pymysql.connect(
        host="localhost",
        user="Oyeniyi_ETL",
        password=password,
        database="electricity_capstone",
        local_infile=1,
        autocommit=True
    )

    try:
        for label, cfg in LOADERS.items():
            chunks = sorted(cfg["dir"].glob("*.csv"))
            print(f"\n=== Loading {label.upper()} ({len(chunks)} files) ===")

            for chunk in chunks:
                print(f"Loading {chunk.name} ...")
                sql = cfg["sql"].format(path=chunk.as_posix())
                with conn.cursor() as cur:
                    cur.execute(sql)
                print("✓ Done")

    except Exception as e:
        print("❌ ERROR:", e)

    finally:
        conn.close()

if __name__ == "__main__":
    main()


Enter your password:  ········



=== Loading DEMAND (5 files) ===
Loading demand.part1.csv ...
